# PerturbDiff inference-only demo

This notebook demonstrates the experimental inference-only API for issue #1: given control cells in an `.h5ad` file and a Tahoe100M drug condition that exists in the released checkpoint vocabulary, generate a predicted expression matrix.

Important limits:
- `finetuned_tahoe100m.ckpt` is a drug-perturbation checkpoint, not a gene-KO checkpoint.
- The perturbation, cell type, batch key, and gene space must map into the checkpoint vocabularies.
- Using a novel CellxGene dataset is an out-of-distribution demo, not a validated biological benchmark.
- The URL below is the dataset provided in issue discussion; its metadata currently describes mostly testis/ovary cells, not skin cells.

In [ ]:
!git clone https://github.com/DeepGraphLearning/PerturbDiff.git
%cd PerturbDiff
!pip -q install anndata omegaconf pytorch-lightning geomloss transformers scikit-learn huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

ckpt = hf_hub_download('katarinayuan/PerturbDiff_release_ckpt', 'finetuned_tahoe100m.ckpt')
data_root = snapshot_download(
    repo_id='katarinayuan/PerturbDiff_data',
    repo_type='dataset',
    allow_patterns=[
        'selected_genes/tahoe100m_real_selected_genes.pkl',
        'meta_data/new_all_emb.pkl',
        'meta_data/idx_to_pertemb.pkl',
        'meta_data/drug_embed_chemberta_cls_dict.pkl',
        'gene_names/*.pkl',
    ],
)
selected_gene_file = f'{data_root}/selected_genes/tahoe100m_real_selected_genes.pkl'
celltype_embedding_path = f'{data_root}/meta_data/new_all_emb.pkl'
pert_embedding_path = f'{data_root}/meta_data/idx_to_pertemb.pkl'
drug_embedding_path = f'{data_root}/meta_data/drug_embed_chemberta_cls_dict.pkl'
gene_embedding_paths = [
    f'{data_root}/gene_names/pbmc_highly_variavle_gene_emb_dict_emb_dict.pkl',
    f'{data_root}/gene_names/replogle_gene_emb_dict_perturbation_emb_dict.pkl',
    f'{data_root}/gene_names/replogle_highly_variavle_gene_emb_dict_emb_dict.pkl',
    f'{data_root}/gene_names/tahoe100m_highly_variavle_gene_emb_dict_emb_dict.pkl',
]

In [3]:
import anndata as ad

h5ad_url = 'https://datasets.cellxgene.cziscience.com/88930108-9c98-43a7-826d-ce454dd0c930.h5ad'
!wget -q -O issue1_demo_input.h5ad {h5ad_url}

adata = ad.read_h5ad('issue1_demo_input.h5ad')
print(adata.shape)
print(adata.obs['cell_type'].value_counts().head())
print(adata.var[['feature_name']].head())

(7276, 60606)
cell_type
spermatid         4009
spermatocyte      3070
male germ cell     116
spermatogonium      64
oocyte              17
Name: count, dtype: int64
                feature_name
ENSG00000000003       TSPAN6
ENSG00000000005         TNMD
ENSG00000000419         DPM1
ENSG00000000457        SCYL3
ENSG00000000460        FIRRM


In [4]:
from src.apps.inference import PerturbDiffPredictor

predictor = PerturbDiffPredictor(
    checkpoint_path=ckpt,
    selected_gene_file=selected_gene_file,
    device='auto',
    covariate_asset_paths={
        'celltype_embedding_path': celltype_embedding_path,
        'gene_embedding_path': gene_embedding_paths,
        'pert_embedding_path': pert_embedding_path,
        'drug_embedding_path': drug_embedding_path,
    },
)

print(predictor.list_perturbations('Dexamethasone', limit=10))
print(predictor.list_cell_types('oocyte', limit=10))
print(predictor.list_cell_types('keratin', limit=10))

[INFO] Applied checkpoint projection replacement for sampling: 12626 -> 2000 genes
["[('Dexamethasone', 0.05, 'uM')]", "[('Dexamethasone', 0.5, 'uM')]", "[('Dexamethasone', 5.0, 'uM')]"]
['oocyte']
['keratinocyte']


In [5]:
# The provided URL contains oocyte cells and `oocyte` is in the checkpoint cell-type vocabulary.
# For a true skin h5ad, replace this subset with a skin cell type such as `keratinocyte` when present.
control = adata[adata.obs['cell_type'].astype(str) == 'oocyte'][:8].copy()

pred = predictor.predict_adata(
    control,
    perturbation="[('Dexamethasone', 0.5, 'uM')]",
    cell_type='oocyte',
    gene_column='feature_name',
    batch_size=4,
    start_time=100,
    normalize_counts=10.0,
    min_gene_overlap=0.8,
    progress=True,
)
pred.write_h5ad('issue1_demo_pred_dexamethasone_oocyte.h5ad')
print(pred.shape, float(pred.X.min()), float(pred.X.max()), float(pred.X.mean()))

(2, 2000) 0.0 3.310044288635254 0.10696197301149368
